In [42]:
import os
import glob
import random
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [45]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

DATA_ROOT = "/home/zahin/Desktop/AI_LAB/datasets/shoe"            # root folder
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
TEST_DIR  = os.path.join(DATA_ROOT, "test")

IMG_SIZE = 224
EPOCHS = 5
BATCH_SIZE = 32

LABEL_MAP = {"left": 0, "right": 1}
CLASS_NAMES = {v: k for k, v in LABEL_MAP.items()}

In [46]:
def _collect_paths_for_split(split_dir):
    paths = []
    labels = []
    for cls_name, lbl in LABEL_MAP.items():
        cls_dir = os.path.join(split_dir, cls_name)
        if not os.path.isdir(cls_dir):
            print(f"Warning: missing folder {cls_dir} (skipping)")
            continue
        for ext in ("jpg","jpeg","png","bmp"):
            paths.extend(glob.glob(os.path.join(cls_dir, f"*.{ext}")))
        labels.extend([lbl] * len(glob.glob(os.path.join(cls_dir, "*.*"))))
    paths = sorted(paths)
    # infer labels by folder name to be robust
    labels = []
    for p in paths:
        parent = os.path.basename(os.path.dirname(p)).lower()
        labels.append(LABEL_MAP.get(parent, -1))
    return paths, np.array(labels, dtype=np.int32)

In [47]:
def load_images_from_paths(paths, expected_size=IMG_SIZE):
    imgs = []
    valid_idx = []
    for i, p in enumerate(paths):
        try:
            im = Image.open(p).convert("RGB")
            if im.size != (expected_size, expected_size):
                print(f"Warning: {p} has size {im.size} - skipping (expected {expected_size}x{expected_size})")
                continue
            arr = np.array(im, dtype=np.uint8)
            imgs.append(arr)
            valid_idx.append(i)
        except Exception as e:
            print(f"Warning: failed to load {p}: {e}")
    if len(imgs) == 0:
        return np.empty((0, expected_size, expected_size, 3), dtype=np.uint8), np.array([], dtype=np.int32), []
    return np.stack(imgs, axis=0), valid_idx

In [48]:
def prepare_split(split_dir):
    paths, labels_all = _collect_paths_for_split(split_dir)
    x_np, valid_idx = load_images_from_paths(paths, expected_size=IMG_SIZE)
    if x_np.shape[0] == 0:
        raise ValueError(f"No valid images found in {split_dir}.")
    y_np = labels_all[valid_idx]
    # display copy scaled 0..1 for plotting
    x_display = x_np.astype(np.float32) / 255.0
    # preprocess for VGG input
    x_pre = preprocess_input(x_np.astype(np.float32))
    return x_pre, x_display, y_np

In [49]:
print("Loading training images...")
x_train_pre_raw, x_train_display_raw, y_train_raw = prepare_split(TRAIN_DIR)
print("Loading test images...")
x_test_pre, x_test_display, y_test = prepare_split(TEST_DIR)

print("Raw counts:")
print(" Train:", x_train_pre_raw.shape, y_train_raw.shape, " Test:", x_test_pre.shape, y_test.shape)

Loading training images...
Loading test images...
Raw counts:
 Train: (736, 224, 224, 3) (736,)  Test: (238, 224, 224, 3) (238,)


In [50]:
# shuffle and split train->train/val (80/20)
perm = np.random.permutation(len(x_train_pre_raw))
x_train_pre_raw = x_train_pre_raw[perm]
x_train_display_raw = x_train_display_raw[perm]
y_train_raw = y_train_raw[perm]

val_idx = int(0.8 * len(x_train_pre_raw))
x_train_pre = x_train_pre_raw[:val_idx]
y_train = y_train_raw[:val_idx]
x_val_pre = x_train_pre_raw[val_idx:]
y_val = y_train_raw[val_idx:]

print("Final dataset shapes:")
print(" Train:", x_train_pre.shape, y_train.shape)
print(" Val:  ", x_val_pre.shape,   y_val.shape)
print(" Test: ", x_test_pre.shape,  y_test.shape)

Final dataset shapes:
 Train: (588, 224, 224, 3) (588,)
 Val:   (148, 224, 224, 3) (148,)
 Test:  (238, 224, 224, 3) (238,)


In [51]:
def build_vgg16_binary(img_size=IMG_SIZE, fine_tune_type="partial", lr=1e-4):
    base = VGG16(weights='imagenet', include_top=False, input_shape=(img_size, img_size, 3))
    if fine_tune_type == "partial":
        # freeze most layers, leave last block trainable
        for layer in base.layers[:-4]:
            layer.trainable = False
        for layer in base.layers[-4:]:
            layer.trainable = True
    elif fine_tune_type == "whole":
        for layer in base.layers:
            layer.trainable = True
    else:
        raise ValueError("fine_tune_type must be 'partial' or 'whole'")

    x = Flatten()(base.output)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    out = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=base.input, outputs=out)
    model.compile(optimizer=Adam(learning_rate=lr),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

In [53]:
print("\nBuilding PARTIAL fine-tune model (most base layers frozen)...")
model_partial = build_vgg16_binary(fine_tune_type="partial")
model_partial.summary(show_trainable=True)


Building PARTIAL fine-tune model (most base layers frozen)...


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┓
┃ Layer (type)                ┃ Output Shape          ┃    Param # ┃ Trai… ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━┩
│ input_layer_4 (InputLayer)  │ (None, 224, 224, 3)   │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block1_conv1 (Conv2D)       │ (None, 224, 224, 64)  │      1,792 │   N   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block1_conv2 (Conv2D)       │ (None, 224, 224, 64)  │     36,928 │   N   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block1_pool (MaxPooling2D)  │ (None, 112, 112, 64)  │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block2_conv1 (Conv2D)       │ (None, 112, 112, 128) │     73,856 │   N   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block2_conv2 (Conv2D)       │ (None, 112, 112, 128) │    147,584 │   N   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block2_pool (MaxPooling2D)  │ (None, 56, 56, 128)   │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block3_conv1 (Conv2D)       │ (None, 56, 56, 256)   │    295,168 │   N   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block3_conv2 (Conv2D)       │ (None, 56, 56, 256)   │    590,080 │   N   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block3_conv3 (Conv2D)       │ (None, 56, 56, 256)   │    590,080 │   N   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block3_pool (MaxPooling2D)  │ (None, 28, 28, 256)   │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block4_conv1 (Conv2D)       │ (None, 28, 28, 512)   │  1,180,160 │   N   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block4_conv2 (Conv2D)       │ (None, 28, 28, 512)   │  2,359,808 │   N   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block4_conv3 (Conv2D)       │ (None, 28, 28, 512)   │  2,359,808 │   N   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block4_pool (MaxPooling2D)  │ (None, 14, 14, 512)   │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block5_conv1 (Conv2D)       │ (None, 14, 14, 512)   │  2,359,808 │   Y   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block5_conv2 (Conv2D)       │ (None, 14, 14, 512)   │  2,359,808 │   Y   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block5_conv3 (Conv2D)       │ (None, 14, 14, 512)   │  2,359,808 │   Y   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block5_pool (MaxPooling2D)  │ (None, 7, 7, 512)     │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ flatten_4 (Flatten)         │ (None, 25088)         │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ dense_8 (Dense)             │ (None, 256)           │  6,422,784 │   Y   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ dropout_4 (Dropout)         │ (None, 256)           │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ dense_9 (Dense)             │ (None, 1)             │        257 │   Y   │
└─────────────────────────────┴───────────────────────┴────────────┴───────┘

 Total params: 21,137,729 (80.63 MB)

 Trainable params: 13,502,465 (51.51 MB)

 Non-trainable params: 7,635,264 (29.13 MB)

In [54]:
history_partial = model_partial.fit(
    x_train_pre, y_train,
    validation_data=(x_val_pre, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

Epoch 1/5
11/19 ━━━━━━━━━━━━━━━━━━━━ 21s 3s/step - accuracy: 0.5355 - loss: 5.5094

KeyboardInterrupt: 

In [56]:
print("\nBuilding WHOLE fine-tune model (all base layers trainable)...")
model_whole = build_vgg16_binary(fine_tune_type="whole")
model_whole.summary(show_trainable=True)


Building WHOLE fine-tune model (all base layers trainable)...


Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┓
┃ Layer (type)                ┃ Output Shape          ┃    Param # ┃ Trai… ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━┩
│ input_layer_6 (InputLayer)  │ (None, 224, 224, 3)   │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block1_conv1 (Conv2D)       │ (None, 224, 224, 64)  │      1,792 │   Y   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block1_conv2 (Conv2D)       │ (None, 224, 224, 64)  │     36,928 │   Y   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block1_pool (MaxPooling2D)  │ (None, 112, 112, 64)  │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block2_conv1 (Conv2D)       │ (None, 112, 112, 128) │     73,856 │   Y   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block2_conv2 (Conv2D)       │ (None, 112, 112, 128) │    147,584 │   Y   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block2_pool (MaxPooling2D)  │ (None, 56, 56, 128)   │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block3_conv1 (Conv2D)       │ (None, 56, 56, 256)   │    295,168 │   Y   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block3_conv2 (Conv2D)       │ (None, 56, 56, 256)   │    590,080 │   Y   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block3_conv3 (Conv2D)       │ (None, 56, 56, 256)   │    590,080 │   Y   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block3_pool (MaxPooling2D)  │ (None, 28, 28, 256)   │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block4_conv1 (Conv2D)       │ (None, 28, 28, 512)   │  1,180,160 │   Y   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block4_conv2 (Conv2D)       │ (None, 28, 28, 512)   │  2,359,808 │   Y   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block4_conv3 (Conv2D)       │ (None, 28, 28, 512)   │  2,359,808 │   Y   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block4_pool (MaxPooling2D)  │ (None, 14, 14, 512)   │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block5_conv1 (Conv2D)       │ (None, 14, 14, 512)   │  2,359,808 │   Y   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block5_conv2 (Conv2D)       │ (None, 14, 14, 512)   │  2,359,808 │   Y   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block5_conv3 (Conv2D)       │ (None, 14, 14, 512)   │  2,359,808 │   Y   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ block5_pool (MaxPooling2D)  │ (None, 7, 7, 512)     │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ flatten_6 (Flatten)         │ (None, 25088)         │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ dense_12 (Dense)            │ (None, 256)           │  6,422,784 │   Y   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ dropout_6 (Dropout)         │ (None, 256)           │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ dense_13 (Dense)            │ (None, 1)             │        257 │   Y   │
└─────────────────────────────┴───────────────────────┴────────────┴───────┘

 Total params: 21,137,729 (80.63 MB)

 Trainable params: 21,137,729 (80.63 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history_whole = model_whole.fit(
    x_train_pre, y_train,
    validation_data=(x_val_pre, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

In [ ]:
def plot_history(hist, title_prefix="Model"):
    h = hist.history
    epochs = range(1, len(h['loss']) + 1)
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    plt.plot(epochs, h['loss'], label='train loss')
    plt.plot(epochs, h['val_loss'], label='val loss')
    plt.title(f"{title_prefix} loss")
    plt.xlabel('epoch'); plt.legend()
    plt.subplot(1,2,2)
    # accuracy key may be 'accuracy' depending on keras version
    acc_key = 'accuracy' if 'accuracy' in h else 'acc'
    val_acc_key = 'val_' + acc_key
    plt.plot(epochs, h[acc_key], label='train acc')
    plt.plot(epochs, h[val_acc_key], label='val acc')
    plt.title(f"{title_prefix} accuracy")
    plt.xlabel('epoch'); plt.legend()
    plt.tight_layout()
    plt.show()

plot_history(history_partial, "Partial fine-tune")
plot_history(history_whole, "Whole fine-tune")

In [ ]:
loss_p, acc_p = model_partial.evaluate(x_test_pre, y_test, verbose=0)
loss_w, acc_w = model_whole.evaluate(x_test_pre, y_test, verbose=0)
print("\nTest evaluation:")
print(f" Partial  -> loss: {loss_p:.4f}, acc: {acc_p:.4f}")
print(f" Whole    -> loss: {loss_w:.4f}, acc: {acc_w:.4f}")

# ----------------------------
# Predict and plot a few test images
# ----------------------------
n_display = 5
rng = np.random.default_rng(SEED)
n_test = len(x_test_pre)
if n_test == 0:
    print("No test images to display.")
else:
    n_display = min(n_display, n_test)
    chosen = rng.choice(n_test, size=n_display, replace=False)

    plt.figure(figsize=(18, 7))
    for i, idx in enumerate(chosen):
        img_disp = x_test_display[idx]        # 0..1 display
        true_label = int(y_test[idx])

        p_partial = float(model_partial.predict(x_test_pre[idx:idx+1], verbose=0)[0][0])
        p_whole   = float(model_whole.predict(x_test_pre[idx:idx+1], verbose=0)[0][0])

        pred_partial = int(p_partial > 0.5)
        pred_whole   = int(p_whole > 0.5)

        ax1 = plt.subplot(2, n_display, 1 + i)
        ax1.imshow(img_disp)
        ax1.axis('off')
        ax1.set_title(f"True: {CLASS_NAMES[true_label]}\nPartial: {CLASS_NAMES[pred_partial]}\n(p={p_partial:.2f})")

        ax2 = plt.subplot(2, n_display, 1 + n_display + i)
        ax2.imshow(img_disp)
        ax2.axis('off')
        ax2.set_title(f"True: {CLASS_NAMES[true_label]}\nWhole: {CLASS_NAMES[pred_whole]}\n(p={p_whole:.2f})")

    plt.tight_layout()
    plt.show()